<a href="https://colab.research.google.com/github/Innovatewithapple/bert-dense-retriever-benchmark/blob/main/EvaluateBeirBenchMark_fiqaDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch datasets beir rank_bm25

In [ ]:
import torch
from transformers import AutoTokenizer,AutoModel
from datasets import load_dataset
from rank_bm25 import BM25Okapi
import numpy as np

In [ ]:
print("--- Loading Custom Model & Tokenizer ---")
device = "cuda" if torch.cuda.is_available() else "cpu"

--- Loading Custom Model & Tokenizer ---


In [ ]:
import torch
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from sentence_transformers import CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"

print("--- 1. Initializing Local Architecture & Checkpoint ---")

model_name = "Innovatewithapple/bert-dense-retriever"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
)
model.to(device)
model.eval()

# --- 2. Define the Custom Wrapper Class ---
# CHANGE 1: BeIR expects 'encode_passages' as the method name, not 'encode_corpus'
class CustomHFModelWrapper:
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    @torch.no_grad()
    def encode_queries(self, queries, batch_size=32, **kwargs):
        embeddings = []
        for i in range(0, len(queries), batch_size):
            batch = queries[i:i+batch_size]
            inputs = self.tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(self.device)
            out = self.model(**inputs)
            embeddings.append(out.cpu())
        return torch.cat(embeddings, dim=0).numpy()

    @torch.no_grad()
    def encode_passages(self, passages, batch_size=32, **kwargs):
        text_passages = [doc["text"] for doc in passages]
        embeddings = []
        for i in range(0, len(text_passages), batch_size):
            batch = text_passages[i:i+batch_size]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(self.device)
            out = self.model(**inputs)
            embeddings.append(out.cpu())
        return torch.cat(embeddings, dim=0).numpy()
    encode_corpus = encode_passages

# --- 3. Stream the FULL 648-Query FiQA Dataset ---
print("\n--- Streaming FULL 648-Query FiQA Dataset from Hugging Face ---")
hf_corpus = load_dataset("BeIR/fiqa", "corpus", split="corpus")
hf_queries = load_dataset("BeIR/fiqa", "queries", split="queries")
hf_qrels = load_dataset("BeIR/fiqa-qrels", split="test")

all_query_ids = set([str(row["query-id"]) for row in hf_qrels])

queries = {str(row["_id"]): row["text"] for row in hf_queries if str(row["_id"]) in all_query_ids}

qrels = {}
needed_corpus_ids = set()
for row in hf_qrels:
    q_id = str(row["query-id"])
    doc_id = str(row["corpus-id"])
    if q_id in all_query_ids:
        if q_id not in qrels:
            qrels[q_id] = {}
        qrels[q_id][doc_id] = int(row["score"])
        needed_corpus_ids.add(doc_id)

corpus = {str(row["_id"]): {"text": row["text"], "title": row["title"]} for row in hf_corpus if str(row["_id"]) in needed_corpus_ids}
print(f"Loaded the COMPLETE dataset: {len(queries)} queries and {len(corpus)} passages.")

# --- 4. Hybrid Retrieval & Re-ranking Loop ---
print("\n--- Phase 1A: Running Vector Search (Dense) ---")
wrapped_model = DRES(CustomHFModelWrapper(model, tokenizer, device), batch_size=16)
dense_results = wrapped_model.search(corpus, queries, top_k=100, score_function="cos_sim")

print("\n--- Phase 1B: Building Local BM25 Index (Sparse - Title Concatenated) ---")
corpus_ids = list(corpus.keys())
# CRUCIAL FIX: Gluing title and text fields together ensures perfect keyword hits
tokenized_corpus = [
    f"{corpus[doc_id]['title']} {corpus[doc_id]['text']}".lower().split(" ")
    for doc_id in corpus_ids
]
bm25 = BM25Okapi(tokenized_corpus)

print("--- Phase 1C: Running Keyword Search (BM25) ---")
sparse_results = {}
for query_id, query_text in queries.items():
    tokenized_query = query_text.lower().split(" ")
    doc_scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(doc_scores)[::-1][:100]
    sparse_results[query_id] = {corpus_ids[idx]: float(doc_scores[idx]) for idx in top_indices}

print("\n--- Phase 2: Merging Candidates & Cross-Encoder Reranking ---")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)
hybrid_reranked_results = {}

for query_id in queries.keys():
    query_text = queries[query_id]
    dense_candidates = list(dense_results.get(query_id, {}).keys())
    sparse_candidates = list(sparse_results.get(query_id, {}).keys())
    combined_candidates = list(set(dense_candidates + sparse_candidates))

    pairs = [[query_text, corpus[doc_id]["text"]] for doc_id in combined_candidates]
    scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)
    hybrid_reranked_results[query_id] = {doc_id: float(score) for doc_id, score in zip(combined_candidates, scores)}

print("\n--- Phase 3: Calculating Optimized Full-Scale Metrics ---")
retriever = EvaluateRetrieval(wrapped_model, score_function="cos_sim")
ndcg, _map, recall, precision = retriever.evaluate(qrels, hybrid_reranked_results, retriever.k_values)

print("\n=============================================")
print("         HYBRID PIPELINE RESULTS (FULL)      ")
print("=============================================")
print(f"NDCG@10:  {ndcg['NDCG@10']:.4f}")
print(f"Recall@10: {recall['Recall@10']:.4f}")
print("=============================================")

--- 1. Initializing Local Architecture & Checkpoint ---
The repository Innovatewithapple/bert-dense-retriever contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/Innovatewithapple/bert-dense-retriever .
 You can inspect the repository content at https://hf.co/Innovatewithapple/bert-dense-retriever.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


--- Streaming FULL 648-Query FiQA Dataset from Hugging Face ---
Loaded the COMPLETE dataset: 648 queries and 1706 passages.

--- Phase 1A: Running Vector Search (Dense) ---

--- Phase 1B: Building Local BM25 Index (Sparse - Title Concatenated) ---
--- Phase 1C: Running Keyword Search (BM25) ---

--- Phase 2: Merging Candidates & Cross-Encoder Reranking ---


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


--- Phase 3: Calculating Optimized Full-Scale Metrics ---

         HYBRID PIPELINE RESULTS (FULL)      
NDCG@10:  0.6150
Recall@10: 0.6683


In [ ]:
print("\n=============================================")
print("         RE-RANKED PIPELINE RESULTS          ")
print("=============================================")
print(f"NDCG@10:  {ndcg['NDCG@10']:.4f}")
print(f"Recall@10: {recall['Recall@10']:.4f}")
print("=============================================")